# 🩺 Medical RAG Chatbot — Build the Retrieval Pipeline

This notebook is the exploratory build behind the project's Flask chatbot: it turns a medical encyclopedia PDF into a Pinecone index
and answers questions with a LangChain retrieval chain.

## Learning Objectives
In this notebook, you will learn:
1. **Loading PDFs** - read every page of `data/Medical_book.pdf` and keep only the metadata retrieval needs
2. **Chunking** - split pages into overlapping 500-character chunks
3. **Embeddings** - embed text locally with the Hugging Face `all-MiniLM-L6-v2` model (384 dimensions)
4. **Pinecone index** - create a serverless index and upsert the chunks once
5. **Retrieval** - query the index for the 3 most similar chunks
6. **RAG chain** - combine retriever, prompt and chat model into a question-answering chain

## Where this fits
`store_index.py` and `app.py` in this project's folder are the scripted versions of these steps: `store_index.py` fills the index,
`app.py` serves the chain behind a Flask chat UI. Here each step runs on its own so you can inspect its output.
The app uses Groq (`openai/gpt-oss-120b`); this notebook uses the course's standard chat model, `gpt-4o-mini`.

## Prerequisites
- Document loaders, text splitters and retrieval chains (repository 03, LangChain fundamentals; repository 05, retrieval and RAG)
- Dependencies installed with `uv sync` in this project's folder
- `PINECONE_API_KEY` and `OPENAI_API_KEY` in this project's `.env`
- The first run downloads the `sentence-transformers/all-MiniLM-L6-v2` model (about 90 MB) from Hugging Face

---
## 📦 Step 1: Environment Setup

Load the keys from this project's `.env` and switch to the project folder, which holds `data/`. Jupyter starts this notebook in `research/`, so the project folder is its parent.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys, locate the project folder, suppress warnings
# ============================================================================
import os
import warnings
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads PINECONE_API_KEY and OPENAI_API_KEY from this project's .env (searched upward from research/)
load_dotenv(find_dotenv(usecwd=True))

# The notebook lives in research/; work from the project folder, like store_index.py and app.py,
# so each document source reads data/Medical_book.pdf (safe to re-run: the target is absolute)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
os.chdir(PROJECT_ROOT)
DATA_DIR = Path("data")

print(f"✅ Environment ready - {len(list(DATA_DIR.glob('*.pdf')))} PDF file(s) in {DATA_DIR}/")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: the chat model that writes answers from retrieved chunks
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 📄 Part 1: Load the PDF and Keep Minimal Metadata

`DirectoryLoader` with `PyPDFLoader` returns one `Document` per PDF page. The loader attaches several metadata fields;
`filter_to_minimal_docs` keeps only `source`, which is all the retrieval chain needs.

In [ ]:
# ============================================================================
# LOAD PDF PAGES: one Document per page
# ============================================================================
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


# Extract text from PDF files
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )

    documents = loader.load()
    return documents

In [ ]:
extracted_docs = load_pdf_files(str(DATA_DIR))

print(f"Loaded {len(extracted_docs)} pages")
print("First page metadata:", extracted_docs[0].metadata)

In [ ]:
# ============================================================================
# MINIMAL METADATA: keep page text + source only
# ============================================================================
from typing import List
from langchain_core.documents import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of Document objects, return a new list of Document objects
    containing only 'source' in metadata and the original page_content.
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [ ]:
minimal_docs = filter_to_minimal_docs(extracted_docs)

print(f"{len(minimal_docs)} documents, metadata now: {minimal_docs[0].metadata}")

---
## ✂️ Part 2: Split Pages into Chunks

Embedding whole pages would blur several topics into one vector. 500-character chunks with a 20-character overlap keep each vector about one idea
without cutting sentences cleanly in half at every boundary.

In [ ]:
# ============================================================================
# CHUNKING: RecursiveCharacterTextSplitter (500 chars, 20 overlap)
# ============================================================================
# Split the documents into smaller chunks
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [ ]:
text_chunk = text_split(minimal_docs)

print(f"Number of chunks: {len(text_chunk)}")
print("Sample chunk:", text_chunk[0].page_content[:200])

---
## 🔢 Part 3: Embed Text with a Local Hugging Face Model

`all-MiniLM-L6-v2` runs on your machine (no API key) and maps any text to a 384-number vector.
That dimension must match the Pinecone index created in Part 4.

In [ ]:
# ============================================================================
# EMBEDDINGS: sentence-transformers/all-MiniLM-L6-v2 (384 dimensions)
# ============================================================================
from langchain_community.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    """
    Download and return the HuggingFace embeddings model.
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embeddings

embedding = download_embeddings()

In [ ]:
vector = embedding.embed_query("What is the capital of France?")

print("Vector length:", len(vector))
print("First 5 values:", [round(v, 4) for v in vector[:5]])

---
## 🌲 Part 4: Create the Pinecone Index and Upsert the Chunks

The index is created once (serverless, cosine similarity, dimension 384). `PineconeVectorStore.from_documents` embeds and upserts every chunk;
because it assigns new ids each time, running it again would store every chunk twice. The cell below upserts only while the index is empty,
then `from_existing_index` connects to whatever the index holds — the same call `app.py` makes.

In [ ]:
# ============================================================================
# PINECONE CLIENT
# ============================================================================
from pinecone import Pinecone

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

print("✅ Pinecone client ready")

In [ ]:
# ============================================================================
# CREATE THE INDEX (once)
# ============================================================================
from pinecone import ServerlessSpec 

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension=384,  # Dimension of the embeddings
        metric= "cosine",  # Cosine similarity
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )


index = pc.Index(index_name)

print(f"✅ Index '{index_name}' ready")

In [ ]:
# ============================================================================
# UPSERT CHUNKS (only while the index is empty, so re-runs never duplicate vectors)
# ============================================================================
from langchain_pinecone import PineconeVectorStore

vector_count = index.describe_index_stats().total_vector_count
if vector_count == 0:
    PineconeVectorStore.from_documents(
        documents=text_chunk,
        embedding=embedding,
        index_name=index_name,
    )
    print(f"✅ Upserted {len(text_chunk)} chunks into '{index_name}'")
else:
    print(f"Index already holds {vector_count} vectors - skipping the upsert")

In [ ]:
# ============================================================================
# CONNECT TO THE EXISTING INDEX (what app.py does at start-up)
# ============================================================================
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

print("✅ Connected to index:", index_name)

---
## 🔎 Part 5: Retrieve Similar Chunks

A retriever wraps the vector store behind one `invoke(question)` call and returns the 3 most similar chunks.

In [ ]:
# ============================================================================
# RETRIEVER: top-3 similarity search
# ============================================================================
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

retrieved_docs = retriever.invoke("What is Acne?")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"[{i}] {doc.page_content[:150]!r}")

The chunks should mention acne directly. A serverless index can take a few seconds to make freshly upserted vectors searchable; if this prints nothing on the very first run, re-run the cell.

---
## 🤖 Part 6: Build the RAG Chain

`create_stuff_documents_chain` puts the retrieved chunks into the system prompt's `{context}` slot and calls the chat model;
`create_retrieval_chain` runs the retriever first and returns both the answer and the chunks it used.
The prompt asks for at most three sentences and an explicit "I don't know" when the context has no answer.

In [ ]:
# ============================================================================
# PROMPT
# ============================================================================
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an Medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [ ]:
# ============================================================================
# RAG CHAIN: retriever -> stuff documents into the prompt -> chat model
# ============================================================================
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

print("✅ RAG chain ready")

In [ ]:
# ============================================================================
# ASK QUESTIONS
# ============================================================================
for question in [
    "what is Acromegaly and gigantism?",
    "what is Acne?",
    "what is the Treatment of Acne?",
]:
    response = rag_chain.invoke({"input": question})
    print(f"Q: {question}\nA: {response['answer']}\n")

---
## 📝 Summary

In this notebook, we learned:

### 1. Loading PDFs
- `DirectoryLoader` + `PyPDFLoader` gives one `Document` per page; trimming metadata to `source` keeps the index lean.

### 2. Chunking
- Small overlapping chunks keep each vector focused on one idea.

### 3. Embeddings
- `all-MiniLM-L6-v2` embeds locally into 384 dimensions, which fixes the Pinecone index dimension.

### 4. Pinecone index
- Create the index once and upsert once: `from_documents` adds new vectors on every call, so guard it.

### 5. Retrieval
- A retriever turns the index into a single `invoke(question)` call returning the top-k chunks.

### 6. RAG chain
- `create_retrieval_chain` + `create_stuff_documents_chain` ground the chat model's answer in the retrieved chunks.

### Next Steps
- **`store_index.py`** and **`app.py`** (the project's app, in this project's folder): the same pipeline as scripts — fill the index, then serve the chain in a Flask chat UI. The project's `README.md` covers Docker and the AWS / Databricks Apps deployments.